In [1]:
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='' # Optional: leave out to create DB first


In [2]:
#!pip install mysql-connector-python --break-system-packages

In [3]:
import mysql.connector
from mysql.connector import Error

In [4]:
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='' # Optional: leave out to create DB first
   )

if connection.is_connected():
    db_info = info = connection.server_info
    print(f"Connected to MySQL Server version: {db_info}")
    cursor = connection.cursor()


Connected to MySQL Server version: 8.4.7


In [5]:

# 2. Create a cursor
cursor = connection.cursor()

# 3. Execute the CREATE DATABASE command
# Use "IF NOT EXISTS" to prevent errors if the DB already exists
cursor.execute("CREATE DATABASE IF NOT EXISTS CHESSBOT")

print("Database created successfully!")

# 4. (Optional) Switch to the new database to start using it
#cursor.execute("USE my_new_database")
# Alternatively, re-assign the database property directly:
# mydb.database = 'my_new_database'


Database created successfully!


In [6]:
# 1. Execute the command
cursor.execute("SHOW DATABASES")

# 2. Fetch and print the results
print("Available Databases:")
for (db_name,) in cursor:
    print(f"- {db_name}")

Available Databases:
- CHESSBOT
- information_schema
- mysql
- performance_schema
- sys


In [7]:
#select database to use
cursor = connection.cursor()
cursor.execute("USE CHESSBOT")
print("Selected CHESSBOT database")

Selected CHESSBOT database


In [8]:
#cursor.execute("DROP TABLE IF EXISTS session")
#cursor.execute("DROP TABLE IF EXISTS moves")
#cursor.execute("DROP TABLE IF EXISTS games")

In [9]:
# --- CREATE: Create a table ---
games_table_name = 'games'
moves_table_name = 'moves'
session_table_name = 'session'

gamestableexists = False
movestableexists = False
sessiontableexists = False

# 1. Execute the command to check if table exists
cursor.execute("SHOW TABLES")

# 2. Fetch and print the results, capture table exists
print("Available Tables:")
for (db_name,) in cursor:
    print(f"- {db_name}")
    if(db_name == games_table_name):
        gamestableexists = True
        print ("-------------")
        print ("table: ",games_table_name," already exists so will not recreate")
    if(db_name == moves_table_name):
        movestableexists = True
        print ("-------------")
        print ("table: ",moves_table_name," already exists so will not recreate")
    if(db_name == session_table_name):
        sessiontableexists = True
        print ("-------------")
        print ("table: ",session_table_name," already exists so will not recreate")


if gamestableexists == False:   # create games table
    print ("-------------")
    print ("table: ",games_table_name," does not exists")
    CreateTable = "CREATE TABLE " + games_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Site VARCHAR(50),White_ELO INT,Black_ELO INT,Opening VARCHAR(100))"
    print(CreateTable)
    cursor.execute(CreateTable)
    print("Table ",games_table_name, " created.")    

if movestableexists == False:   # create moves table
    print ("-------------")
    print ("table: ",moves_table_name," does not exists")
    CreateTable2 = "CREATE TABLE " + moves_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Set5 VARCHAR(75),Set10 VARCHAR(75),Set15 VARCHAR(75),Set20 VARCHAR(75),Set30 VARCHAR(150),Set40 VARCHAR(150),Set70 VARCHAR(450),Set_Remain VARCHAR(450),BlackELO INT,Game_id INT,FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable2)
    cursor.execute(CreateTable2)
    print("Table ",moves_table_name, " created.")    

    ### create Move indexes
    cursor.execute("CREATE INDEX idx_set5 ON moves(Set5)")
    cursor.execute("CREATE INDEX idx_set10 ON moves(Set10)")
    cursor.execute("CREATE INDEX idx_set15 ON moves(Set15)")
    cursor.execute("CREATE INDEX idx_set20 ON moves(Set20)")
    cursor.execute("CREATE INDEX idx_set30 ON moves(Set30)")
    print ("-------------")
    print ("Moves table indexes are created")

if sessiontableexists == False:   # create session table
    print ("-------------")
    print ("table: ",session_table_name," does not exists")
    CreateTable3 = "CREATE TABLE " + session_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Cookie VARCHAR(255),Date_Time DATETIME,Game_id INT,Param_colour VARCHAR(50),Param_difficult VARCHAR(50),Param_metric VARCHAR(50),Param_snark VARCHAR(50),progress INT,result VARCHAR(255),FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable3)
    cursor.execute(CreateTable3)
    print("Table ",session_table_name, " created.")    


Available Tables:
- games
-------------
table:  games  already exists so will not recreate
- moves
-------------
table:  moves  already exists so will not recreate
- session
-------------
table:  session  already exists so will not recreate


In [10]:
#check games table

cursor.execute("DESCRIBE "+games_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Site                 varchar(50)     YES             
White_ELO            int             YES             
Black_ELO            int             YES             
Opening              varchar(100)    YES             


In [11]:
#check moves table

cursor.execute("DESCRIBE "+moves_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Set5                 varchar(75)     YES        MUL  
Set10                varchar(75)     YES        MUL  
Set15                varchar(75)     YES        MUL  
Set20                varchar(75)     YES        MUL  
Set30                varchar(150)    YES        MUL  
Set40                varchar(150)    YES             
Set70                varchar(450)    YES             
Set_Remain           varchar(450)    YES             
BlackELO             int             YES             
Game_id              int             YES        MUL  


In [12]:
#check session table

cursor.execute("DESCRIBE "+session_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Cookie               varchar(255)    YES             
Date_Time            datetime        YES             
Game_id              int             YES        MUL  
Param_colour         varchar(50)     YES             
Param_difficult      varchar(50)     YES             
Param_metric         varchar(50)     YES             
Param_snark          varchar(50)     YES             
progress             int             YES             
result               varchar(255)    YES             


In [13]:
# --- READ: Fetch the record count of games table ---
cursor.execute(f"SELECT COUNT(*) FROM {games_table_name}")
count = cursor.fetchone()[0]
print(f"{games_table_name} table has entries: {count}")


games table has entries: 27126


In [14]:
# --- READ: Fetch the record count of moves table ---
cursor.execute(f"SELECT COUNT(*) FROM {moves_table_name}")
count = cursor.fetchone()[0]
print(f"{moves_table_name} table has entries: {count}")


moves table has entries: 27126


In [15]:
# --- READ: Fetch the record count of session table ---
cursor.execute(f"SELECT COUNT(*) FROM {session_table_name}")
count = cursor.fetchone()[0]
print(f"{session_table_name} table has entries: {count}")


session table has entries: 0


In [16]:
#Michael's sandbox..
#INSERT

insert_query = "INSERT INTO " + games_table_name + "(Site, White_ELO,Black_ELO,Opening) VALUES (%s, %s, %s, %s)"
cursor.execute(insert_query, ("https://lichess.org/gx3qb4ur", 1155,1096,"King's Indian Attack"))
connection.commit()  # Required to save changes
InsertedRecord = cursor.lastrowid
print(f"Record inserted. ID: {InsertedRecord}")


Record inserted. ID: 27131


In [29]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM moves WHERE  Set5 = '1. d2d4 d7d5 2. c2c4 e7e6 3. c4c5 c7c6 4. b2b4 a7a5 5. a2a3 a5b4' AND Set10 = '6. e2e3 d8a5 7. c1d2 b8d7 8. d2b4 a5a7 9. g1f3 g8f6 10. f1e2 h7h6' AND Set15 = '11. e1g1 b7b6 12. c5b6 a7b6 13. b4f8 d7f8 14. b1c3 f8d7 15. a1b1 b6c7' AND Set20 = '16. b1b3 f6g4 17. g2g3 g7g5 18. d1d2 d7f6 19. f3e1 e6e5 20. d4e5 c7e5' AND Set30 = '21. d2d4 e5d4 22. e3d4 f6e4 23. c3e4 d5e4 24. e2g4 c8g4 25. e1g2 g4e6 26. b3b7 a8a3 27. b7b8 e8e7 28. b8h8 e6c4 29. f1c1 c4d5 30. g2e3 d5b3' AND Set40 = '31. c1c6 a3a1 32. g1g2 b3a4 33. c6h6 a4b5 34. h6h5 g5g4 35. h5e5 e7d6 36. e5b5 d6e6 37. h8h6 e6e7 38. b5e5 e7d7 39. e5e4 f7f5 40. e4e5 a1a8' AND Set70 LIKE '41. e5f5 d7e7 42. f5b5 e7d7 43. b5b6 a8c8 44. e3g4 d7e7 45. b6b7%' "
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 1
---------------
18
1. d2d4 d7d5 2. c2c4 e7e6 3. c4c5 c7c6 4. b2b4 a7a5 5. a2a3 a5b4
6. e2e3 d8a5 7. c1d2 b8d7 8. d2b4 a5a7 9. g1f3 g8f6 10. f1e2 h7h6
11. e1g1 b7b6 12. c5b6 a7b6 13. b4f8 d7f8 14. b1c3 f8d7 15. a1b1 b6c7
16. b1b3 f6g4 17. g2g3 g7g5 18. d1d2 d7f6 19. f3e1 e6e5 20. d4e5 c7e5
21. d2d4 e5d4 22. e3d4 f6e4 23. c3e4 d5e4 24. e2g4 c8g4 25. e1g2 g4e6 26. b3b7 a8a3 27. b7b8 e8e7 28. b8h8 e6c4 29. f1c1 c4d5 30. g2e3 d5b3
31. c1c6 a3a1 32. g1g2 b3a4 33. c6h6 a4b5 34. h6h5 g5g4 35. h5e5 e7d6 36. e5b5 d6e6 37. h8h6 e6e7 38. b5e5 e7d7 39. e5e4 f7f5 40. e4e5 a1a8
41. e5f5 d7e7 42. f5b5 e7d7 43. b5b6 a8c8 44. e3g4 d7e7 45. b6b7 e7d8 46. h6h8
None
1550
19


In [25]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM games WHERE  id = '35'"
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 1
---------------
35
https://lichess.org/k20znary
1698
1553
Nimzo-Larsen Attack


In [18]:
#Michael's sandbox..
#UPDATE

update_query = "UPDATE games SET Opening = %s WHERE id = %s"
cursor.execute(update_query, ("Some new death method", InsertedRecord))
connection.commit()
print("Record updated.")


Record updated.


In [19]:
#Michael's sandbox..
#DELETE

delete_query = "DELETE FROM games WHERE id = %s"
cursor.execute(delete_query, (InsertedRecord,))
connection.commit()
print("Record deleted.")


Record deleted.
